In [1]:
import pandas as pd
from pathlib import Path


# ============================================================
# STEP 10 - BUILD COMPLETE ATTENDANCE REQUIRED MASTER
# ============================================================
# Grain:
#   attendance_required = one employee x one calendar date
#   dim_employee        = one row per employee_id
#   fact_employee_week  = one employee x one calendar week
#
# Important:
#   - Employee master is extracted BEFORE building the calendar grid.
#   - Employee attributes are retained dynamically (not manually
#     restricted to a short list of columns).
#   - The employee/date grid is built from dim_employee x calendar.
#   - Attendance is LEFT JOINED onto that grid. Employees/dates
#     without attendance transactions therefore remain in the output.
# ============================================================


# ============================================================
# 1. CONFIGURATION
# ============================================================

ATTENDANCE_FILE = Path(
    "output/second_reports/attendance_feature_engineered.xlsx"
)

CALENDAR_FILE = Path(
    "output/dim_calendar.parquet"
)

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMPLOYEE_PARQUET = OUTPUT_DIR / "dim_employee.parquet"
EMPLOYEE_CSV = OUTPUT_DIR / "dim_employee.csv"

REQUIRED_PARQUET = OUTPUT_DIR / "attendance_required.parquet"
REQUIRED_CSV = OUTPUT_DIR / "attendance_required.csv"

DAILY_PARQUET = OUTPUT_DIR / "fact_attendance_daily.parquet"
DAILY_CSV = OUTPUT_DIR / "fact_attendance_daily.csv"

WEEKLY_PARQUET = OUTPUT_DIR / "fact_employee_week.parquet"
WEEKLY_CSV = OUTPUT_DIR / "fact_employee_week.csv"


# ============================================================
# 2. LOAD INPUT DATA
# ============================================================

print("Loading attendance and calendar data...")

if not ATTENDANCE_FILE.exists():
    raise FileNotFoundError(
        f"Attendance file not found: {ATTENDANCE_FILE}"
    )

if not CALENDAR_FILE.exists():
    raise FileNotFoundError(
        f"Calendar file not found: {CALENDAR_FILE}"
    )

attendance = pd.read_excel(ATTENDANCE_FILE)
calendar = pd.read_parquet(CALENDAR_FILE)

print("Attendance source rows:", len(attendance))
print("Calendar rows:", len(calendar))


# ============================================================
# 3. VALIDATE REQUIRED INPUT COLUMNS
# ============================================================

# These three flags were already created in the previous steps.
# Step 10 uses them; it does not recreate or rename them.

attendance_required_columns = [
    "employee_id",
    "attendance_date",
    "cost_center_type",
    "weekly_work_quota",
    "has_clock_in",
    "has_clock_out",
    "has_attendance_activity"
]

calendar_required_columns = [
    "attendance_date",
    "week_start",
    "week_end",
    "is_weekend",
    "is_national_holiday",
    "is_joint_leave",
    "weekday_national_holidays",
    "weekday_joint_leaves"
]

missing_attendance = [
    col for col in attendance_required_columns
    if col not in attendance.columns
]

missing_calendar = [
    col for col in calendar_required_columns
    if col not in calendar.columns
]

if missing_attendance:
    raise ValueError(
        "Missing required attendance columns: "
        f"{missing_attendance}"
    )

if missing_calendar:
    raise ValueError(
        "Missing required calendar columns: "
        f"{missing_calendar}"
    )


# ============================================================
# 4. STANDARDIZE KEYS, DATES, AND FLAGS
# ============================================================

attendance["employee_id"] = (
    attendance["employee_id"]
    .astype("string")
    .str.strip()
)

attendance["attendance_date"] = pd.to_datetime(
    attendance["attendance_date"],
    errors="coerce"
).dt.normalize()

calendar["attendance_date"] = pd.to_datetime(
    calendar["attendance_date"],
    errors="coerce"
).dt.normalize()

calendar["week_start"] = pd.to_datetime(
    calendar["week_start"],
    errors="coerce"
).dt.normalize()

calendar["week_end"] = pd.to_datetime(
    calendar["week_end"],
    errors="coerce"
).dt.normalize()

# Employment dates are optional. If present, they are used to
# restrict the employee x calendar combinations to employment dates.
for col in ["employee_start_date", "employee_end_date"]:
    if col in attendance.columns:
        attendance[col] = pd.to_datetime(
            attendance[col],
            errors="coerce"
        ).dt.normalize()


def to_boolean(series):
    """Convert common Excel/Pandas boolean representations safely."""

    true_values = {"TRUE", "1", "YES", "Y", "T"}
    false_values = {"FALSE", "0", "NO", "N", "F", "", "NONE", "NAN", "<NA>"}

    def convert(value):
        if pd.isna(value):
            return False

        if isinstance(value, bool):
            return value

        value_text = str(value).strip().upper()

        if value_text in true_values:
            return True

        if value_text in false_values:
            return False

        raise ValueError(
            f"Unexpected boolean value: {value!r}"
        )

    return series.map(convert).astype(bool)


for col in [
    "has_clock_in",
    "has_clock_out",
    "has_attendance_activity"
]:
    attendance[col] = to_boolean(attendance[col])

for col in [
    "is_weekend",
    "is_national_holiday",
    "is_joint_leave"
]:
    calendar[col] = to_boolean(calendar[col])

# Basic key validation
if attendance["employee_id"].isna().any():
    raise ValueError("Missing employee_id found in attendance data.")

if attendance["employee_id"].eq("").any():
    raise ValueError("Empty employee_id found in attendance data.")

if attendance["attendance_date"].isna().any():
    raise ValueError("Invalid attendance_date found in attendance data.")

if calendar["attendance_date"].isna().any():
    raise ValueError("Invalid attendance_date found in calendar.")

if calendar["attendance_date"].duplicated().any():
    raise ValueError(
        "Calendar must contain exactly one row per attendance_date."
    )

if calendar["week_start"].isna().any():
    raise ValueError("Missing or invalid week_start in calendar.")

if calendar["week_end"].isna().any():
    raise ValueError("Missing or invalid week_end in calendar.")


# ============================================================
# 5. PREPARE COST CENTER CLASSIFICATION
# ============================================================

attendance["cost_center_type"] = (
    attendance["cost_center_type"]
    .astype("string")
    .str.strip()
    .str.upper()
    .fillna("UNKNOWN")
)

if "schedule_type" in attendance.columns:
    attendance["schedule_type"] = (
        attendance["schedule_type"]
        .astype("string")
        .str.strip()
        .str.upper()
        .fillna("UNKNOWN")
    )

attendance["weekly_work_quota"] = pd.to_numeric(
    attendance["weekly_work_quota"],
    errors="coerce"
).astype("Int64")

valid_cost_center_types = {
    "FIXED_5",
    "NON_FIXED_6",
    "UNKNOWN"
}

unexpected_types = (
    set(attendance["cost_center_type"].dropna().unique())
    - valid_cost_center_types
)

if unexpected_types:
    raise ValueError(
        "Unexpected cost_center_type values: "
        f"{sorted(unexpected_types)}"
    )

# Validate quota rules already assigned by previous steps.
invalid_fixed_quota = (
    attendance["cost_center_type"].eq("FIXED_5")
    & attendance["weekly_work_quota"].ne(5)
)

invalid_non_fixed_quota = (
    attendance["cost_center_type"].eq("NON_FIXED_6")
    & attendance["weekly_work_quota"].ne(6)
)

if invalid_fixed_quota.any():
    raise ValueError(
        "FIXED_5 employees must have weekly_work_quota = 5."
    )

if invalid_non_fixed_quota.any():
    raise ValueError(
        "NON_FIXED_6 employees must have weekly_work_quota = 6."
    )


# ============================================================
# 6. BUILD EMPLOYEE MASTER FROM ALL AVAILABLE EMPLOYEE DATA
# ============================================================

# Columns describing an attendance event are not employee-master
# attributes. All other source columns are retained as employee
# attributes, including department, division, organization, location,
# store, cost center, employee dates, and any other available fields.

attendance_event_columns = {
    "attendance_date",
    "clock_in_time",
    "clock_out_time",
    "clock_in_time_be",
    "clock_out_time_be",
    "attendance_in",
    "attendance_out",
    "shift_in",
    "shift_out",
    "shift_code",
    "attendance_status",
    "attendance_record_count",
    "has_clock_in",
    "has_clock_out",
    "has_attendance_activity",
    "is_expected_workday",
    "daily_work_requirement",
    "year_week",
    "week_start",
    "week_end",
    "is_weekend",
    "is_national_holiday",
    "is_joint_leave",
    "weekday_national_holidays",
    "weekday_joint_leaves"
}

employee_attribute_columns = [
    col
    for col in attendance.columns
    if col not in attendance_event_columns
    and col != "employee_id"
]

# The employee attributes are selected by the most frequent non-null
# value per employee. This prevents arbitrary row selection and keeps
# the full available employee information in the master.
#
# If an attribute changes over time for one employee, the conflict is
# reported below. A true historical organization transfer requires an
# effective-dated employee master; it cannot be reconstructed reliably
# from a single employee-level value.

def most_frequent_non_null(series):
    values = series.dropna()

    if values.empty:
        return pd.NA

    counts = values.value_counts(dropna=True)
    highest_count = counts.max()
    candidates = set(counts[counts == highest_count].index.tolist())

    # Deterministic tie-break: use the last observed value among
    # the tied candidates.
    for value in reversed(values.tolist()):
        if value in candidates:
            return value

    return values.iloc[-1]


employee_aggregation = {
    col: most_frequent_non_null
    for col in employee_attribute_columns
}

employee_master = (
    attendance[
        ["employee_id"] + employee_attribute_columns
    ]
    .groupby(
        "employee_id",
        as_index=False,
        dropna=False
    )
    .agg(employee_aggregation)
)

if employee_master["employee_id"].duplicated().any():
    raise ValueError(
        "Employee master contains duplicate employee_id."
    )

# Normalize employee-level dates after aggregation.
for col in ["employee_start_date", "employee_end_date"]:
    if col in employee_master.columns:
        employee_master[col] = pd.to_datetime(
            employee_master[col],
            errors="coerce"
        ).dt.normalize()

print("\nEmployee master created.")
print("Unique employees:", len(employee_master))
print("Employee attributes retained:", len(employee_attribute_columns))


# ============================================================
# 7. REPORT EMPLOYEE ATTRIBUTE CONFLICTS
# ============================================================

conflict_rows = []

for col in employee_attribute_columns:
    counts = (
        attendance.groupby("employee_id")[col]
        .nunique(dropna=True)
    )

    conflicting_ids = counts[counts > 1]

    if not conflicting_ids.empty:
        conflict_rows.append({
            "column": col,
            "employees_with_multiple_values": int(
                len(conflicting_ids)
            )
        })

employee_conflicts = pd.DataFrame(
    conflict_rows,
    columns=[
        "column",
        "employees_with_multiple_values"
    ]
)

if not employee_conflicts.empty:
    print("\nWARNING: Employee attributes have multiple values.")
    print(employee_conflicts.to_string(index=False))
    print(
        "The employee master uses the most frequent non-null value. "
        "Review these conflicts, especially department, division, "
        "location, and cost center."
    )
else:
    print("Employee attribute consistency: VALID")


# ============================================================
# 8. PREPARE DAILY ATTENDANCE FACT
# ============================================================

# Keep original attendance detail fields where available.
# These fields are aggregated to one employee-date row.
daily_detail_candidates = [
    "clock_in_time",
    "clock_out_time",
    "clock_in_time_be",
    "clock_out_time_be",
    "attendance_in",
    "attendance_out",
    "shift_in",
    "shift_out",
    "shift_code",
    "attendance_status"
]

daily_detail_columns = [
    col for col in daily_detail_candidates
    if col in attendance.columns
]


def first_non_null(series):
    values = series.dropna()

    if values.empty:
        return pd.NA

    return values.iloc[0]


daily_aggregation = {
    "attendance_record_count": (
        "attendance_date",
        "size"
    ),
    "has_clock_in": (
        "has_clock_in",
        "max"
    ),
    "has_clock_out": (
        "has_clock_out",
        "max"
    ),
    "has_attendance_activity": (
        "has_attendance_activity",
        "max"
    )
}

for col in daily_detail_columns:
    daily_aggregation[col] = (
        col,
        first_non_null
    )

daily_attendance = (
    attendance.groupby(
        ["employee_id", "attendance_date"],
        as_index=False,
        dropna=False
    )
    .agg(**daily_aggregation)
)

if daily_attendance.duplicated(
    ["employee_id", "attendance_date"]
).any():
    raise ValueError(
        "Daily attendance contains duplicate employee-date keys."
    )

print("\nDaily attendance fact created.")
print("Unique employee-date transactions:", len(daily_attendance))


# ============================================================
# 9. PREPARE CALENDAR REFERENCE
# ============================================================

calendar_columns = [
    "attendance_date",
    "week_start",
    "week_end",
    "is_weekend",
    "is_national_holiday",
    "is_joint_leave",
    "weekday_national_holidays",
    "weekday_joint_leaves"
]

calendar_reference = calendar[
    calendar_columns
].copy()

for col in [
    "weekday_national_holidays",
    "weekday_joint_leaves"
]:
    calendar_reference[col] = pd.to_numeric(
        calendar_reference[col],
        errors="coerce"
    ).fillna(0).astype("int64")

# Verify that every attendance transaction date exists in calendar.
calendar_dates = set(calendar_reference["attendance_date"])
attendance_dates = set(daily_attendance["attendance_date"])

unmatched_dates = attendance_dates - calendar_dates

if unmatched_dates:
    raise ValueError(
        "Attendance dates not found in calendar. "
        f"Examples: {sorted(unmatched_dates)[:10]}"
    )


# ============================================================
# 10. CREATE COMPLETE EMPLOYEE x CALENDAR GRID
# ============================================================

# This is the most important correction:
# DO NOT start from attendance transactions.
# Start from all employees x all calendar dates.
#
# Therefore an employee remains in the output even when there is
# no attendance transaction for a particular date.

print("\nCreating employee x calendar combinations...")

attendance_required = employee_master.merge(
    calendar_reference,
    how="cross"
)

# Apply employment date boundaries only when the corresponding
# employee date columns are available.
if "employee_start_date" in attendance_required.columns:
    attendance_required = attendance_required.loc[
        attendance_required["employee_start_date"].isna()
        | (
            attendance_required["attendance_date"]
            >= attendance_required["employee_start_date"]
        )
    ].copy()

if "employee_end_date" in attendance_required.columns:
    attendance_required = attendance_required.loc[
        attendance_required["employee_end_date"].isna()
        | (
            attendance_required["attendance_date"]
            <= attendance_required["employee_end_date"]
        )
    ].copy()

# LEFT JOIN actual attendance. No-transaction dates remain.
attendance_required = attendance_required.merge(
    daily_attendance,
    on=["employee_id", "attendance_date"],
    how="left",
    validate="one_to_one"
)

# Fill transaction counts and activity flags for dates without
# an attendance transaction.
attendance_required["attendance_record_count"] = (
    attendance_required["attendance_record_count"]
    .fillna(0)
    .astype("int64")
)

for col in [
    "has_clock_in",
    "has_clock_out",
    "has_attendance_activity"
]:
    attendance_required[col] = (
        attendance_required[col]
        .fillna(False)
        .astype(bool)
    )

attendance_required["attendance_record_status"] = (
    attendance_required["attendance_record_count"]
    .gt(0)
    .map({
        True: "HAS_ATTENDANCE_RECORD",
        False: "NO_ATTENDANCE_RECORD"
    })
)

print("Employee x calendar rows:", len(attendance_required))
print(
    "Rows without attendance transactions:",
    int(
        attendance_required["attendance_record_count"].eq(0).sum()
    )
)


# ============================================================
# 11. CALCULATE DAILY WORK REQUIREMENT
# ============================================================

# Business rules retained from the existing Step 10:
# FIXED_5:
#   Monday-Friday, excluding national holidays.
# NON_FIXED_6:
#   Daily scheduled workday is unknown; weekly quota remains six.
#
# Joint leave is retained as a calendar attribute. It is not
# subtracted from FIXED_5 quota in this version because the existing
# Step 10 rule subtracts weekday national holidays only.

attendance_required["cost_center_type"] = (
    attendance_required["cost_center_type"]
    .astype("string")
    .str.strip()
    .str.upper()
    .fillna("UNKNOWN")
)

attendance_required["is_expected_workday"] = pd.Series(
    pd.NA,
    index=attendance_required.index,
    dtype="boolean"
)

attendance_required["daily_work_requirement"] = pd.Series(
    pd.NA,
    index=attendance_required.index,
    dtype="Int64"
)

is_fixed = attendance_required["cost_center_type"].eq("FIXED_5")
is_weekday = ~attendance_required["is_weekend"]

is_fixed_workday = (
    is_fixed
    & is_weekday
    & ~attendance_required["is_national_holiday"]
)

attendance_required.loc[
    is_fixed,
    "is_expected_workday"
] = is_fixed_workday.loc[is_fixed]

attendance_required.loc[
    is_fixed_workday,
    "daily_work_requirement"
] = 1

attendance_required.loc[
    is_fixed & ~is_fixed_workday,
    "daily_work_requirement"
] = 0


# ============================================================
# 12. BUILD WEEKLY CALENDAR REFERENCE
# ============================================================

weekly_calendar = (
    calendar_reference.groupby(
        "week_start",
        as_index=False
    )
    .agg(
        week_end=("week_end", "max"),
        calendar_days=("attendance_date", "nunique"),
        weekday_national_holidays=(
            "weekday_national_holidays",
            "max"
        ),
        weekday_joint_leaves=(
            "weekday_joint_leaves",
            "max"
        )
    )
)

weekly_calendar["is_full_calendar_week"] = (
    weekly_calendar["calendar_days"].eq(7)
)


# ============================================================
# 13. AGGREGATE COMPLETE EMPLOYEE-WEEK DATA
# ============================================================

# Since attendance_required already contains every employee/date,
# weekly aggregation now includes weeks with zero attendance activity.

attendance_required["expected_workday_attended"] = (
    attendance_required["is_expected_workday"].fillna(False)
    & attendance_required["has_attendance_activity"]
)

attendance_required["attendance_outside_expected_day"] = (
    is_fixed
    & ~attendance_required["is_expected_workday"].fillna(False)
    & attendance_required["has_attendance_activity"]
)

weekly = (
    attendance_required.groupby(
        ["employee_id", "week_start"],
        as_index=False,
        dropna=False
    )
    .agg(
        attendance_record_count=(
            "attendance_record_count",
            "sum"
        ),
        attendance_days_observed=(
            "attendance_date",
            "nunique"
        ),
        actual_attendance_days=(
            "has_attendance_activity",
            "sum"
        ),
        actual_clock_in_days=(
            "has_clock_in",
            "sum"
        ),
        actual_clock_out_days=(
            "has_clock_out",
            "sum"
        ),
        actual_expected_workdays_attended=(
            "expected_workday_attended",
            "sum"
        ),
        attendance_outside_expected_days=(
            "attendance_outside_expected_day",
            "sum"
        )
    )
)

weekly = weekly.merge(
    weekly_calendar,
    on="week_start",
    how="left",
    validate="many_to_one"
)

# Add all employee master attributes to weekly output.
weekly = weekly.merge(
    employee_master,
    on="employee_id",
    how="left",
    validate="many_to_one"
)

if weekly["calendar_days"].isna().any():
    raise ValueError(
        "Some employee weeks have no calendar reference."
    )


# ============================================================
# 14. CALCULATE WEEKLY REQUIRED DAYS
# ============================================================

weekly["cost_center_type"] = (
    weekly["cost_center_type"]
    .astype("string")
    .str.strip()
    .str.upper()
    .fillna("UNKNOWN")
)

weekly["weekly_required_days"] = pd.Series(
    pd.NA,
    index=weekly.index,
    dtype="Int64"
)

full_week = weekly["is_full_calendar_week"]
fixed_mask = weekly["cost_center_type"].eq("FIXED_5")
non_fixed_mask = weekly["cost_center_type"].eq("NON_FIXED_6")

# FIXED_5:
# Five working days minus weekday national holidays.
fixed_required = (
    5 - weekly["weekday_national_holidays"]
)

if (fixed_required < 0).any():
    raise ValueError(
        "Weekday national holidays exceed the five-day quota."
    )

weekly.loc[
    full_week & fixed_mask,
    "weekly_required_days"
] = fixed_required.loc[
    full_week & fixed_mask
].astype("int64")

# NON_FIXED_6:
# Weekly quota remains six days, including holiday weeks.
weekly.loc[
    full_week & non_fixed_mask,
    "weekly_required_days"
] = 6


# ============================================================
# 15. CALCULATE WEEKLY ATTENDANCE GAP
# ============================================================

weekly["attendance_gap"] = pd.Series(
    pd.NA,
    index=weekly.index,
    dtype="Int64"
)

fixed_gap = (
    weekly["weekly_required_days"]
    - weekly["actual_expected_workdays_attended"]
)

weekly.loc[
    full_week & fixed_mask,
    "attendance_gap"
] = fixed_gap.loc[
    full_week & fixed_mask
]

non_fixed_gap = (
    weekly["weekly_required_days"]
    - weekly["actual_attendance_days"]
)

weekly.loc[
    full_week & non_fixed_mask,
    "attendance_gap"
] = non_fixed_gap.loc[
    full_week & non_fixed_mask
]


# ============================================================
# 16. CREATE WEEKLY ATTENDANCE STATUS
# ============================================================

weekly["weekly_attendance_status"] = "UNKNOWN"

weekly.loc[
    ~full_week,
    "weekly_attendance_status"
] = "PARTIAL_WEEK_NOT_EVALUATED"

unknown_mask = weekly["cost_center_type"].eq("UNKNOWN")

weekly.loc[
    full_week & unknown_mask,
    "weekly_attendance_status"
] = "UNKNOWN_COST_CENTER"

weekly.loc[
    full_week
    & fixed_mask
    & (weekly["attendance_gap"] <= 0),
    "weekly_attendance_status"
] = "QUOTA_MET_OR_EXCEEDED"

weekly.loc[
    full_week
    & fixed_mask
    & (weekly["attendance_gap"] > 0),
    "weekly_attendance_status"
] = "BELOW_REQUIRED_DAYS"

weekly.loc[
    full_week
    & non_fixed_mask
    & (weekly["attendance_gap"] <= 0),
    "weekly_attendance_status"
] = "AT_OR_ABOVE_WEEKLY_QUOTA"

weekly.loc[
    full_week
    & non_fixed_mask
    & (weekly["attendance_gap"] > 0),
    "weekly_attendance_status"
] = "BELOW_WEEKLY_QUOTA_REVIEW"


# ============================================================
# 17. CREATE WEEKLY INDICATORS
# ============================================================

weekly["is_quota_gap"] = (
    weekly["attendance_gap"] > 0
).astype("boolean")

weekly["is_quota_met"] = (
    weekly["attendance_gap"] <= 0
).astype("boolean")

not_evaluated = (
    ~full_week
    | unknown_mask
)

weekly.loc[
    not_evaluated,
    ["is_quota_gap", "is_quota_met"]
] = pd.NA


# ============================================================
# 18. DATA VALIDATION
# ============================================================

print("\n==========================================")
print("STEP 10 DATA VALIDATION")
print("==========================================")

# Employee master grain
if employee_master["employee_id"].duplicated().any():
    raise ValueError("Duplicate employee_id in dim_employee.")

# Attendance required grain
if attendance_required.duplicated(
    ["employee_id", "attendance_date"]
).any():
    raise ValueError(
        "Duplicate employee-date rows in attendance_required."
    )

# Weekly grain
if weekly.duplicated(
    ["employee_id", "week_start"]
).any():
    raise ValueError(
        "Duplicate employee-week rows in weekly fact."
    )

# Every employee in employee master must appear in attendance_required
employee_ids_master = set(employee_master["employee_id"])
employee_ids_required = set(attendance_required["employee_id"])

missing_employees = employee_ids_master - employee_ids_required

if missing_employees:
    raise ValueError(
        "Some employees are missing from attendance_required: "
        f"{list(missing_employees)[:10]}"
    )

# No attendance transaction count should be negative.
if attendance_required["attendance_record_count"].lt(0).any():
    raise ValueError("Negative attendance_record_count found.")

# Activity flags must be consistent.
invalid_activity = (
    attendance_required["has_attendance_activity"]
    != (
        attendance_required["has_clock_in"]
        | attendance_required["has_clock_out"]
    )
)

if invalid_activity.any():
    raise ValueError(
        "has_attendance_activity is inconsistent with clock flags."
    )

# Validate weekly quota rules.
if (
    weekly.loc[
        weekly["cost_center_type"].eq("FIXED_5"),
        "weekly_required_days"
    ].dropna() > 5
).any():
    raise ValueError("FIXED_5 weekly quota exceeds five days.")

if (
    weekly.loc[
        weekly["cost_center_type"].eq("NON_FIXED_6"),
        "weekly_required_days"
    ].dropna() != 6
).any():
    raise ValueError("NON_FIXED_6 weekly quota must equal six days.")

print("Employee master grain: VALID")
print("Attendance required grain: VALID")
print("Weekly grain: VALID")
print("Employee coverage: VALID")
print("Attendance activity flags: VALID")
print("Weekly quota rules: VALID")


# ============================================================
# 19. EXPORT EMPLOYEE MASTER
# ============================================================

employee_master.to_parquet(
    EMPLOYEE_PARQUET,
    index=False
)

employee_master.to_csv(
    EMPLOYEE_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 20. EXPORT COMPLETE ATTENDANCE REQUIRED
# ============================================================

# This is the main output for the dashboard:
# all employee attributes + all calendar dates in employment period
# + attendance transaction fields + daily requirement fields.

attendance_required.to_parquet(
    REQUIRED_PARQUET,
    index=False
)

attendance_required.to_csv(
    REQUIRED_CSV,
    index=False,
    encoding="utf-8-sig"
)


# Keep the existing daily output names for compatibility.
attendance_required.to_parquet(
    DAILY_PARQUET,
    index=False
)

attendance_required.to_csv(
    DAILY_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 21. EXPORT WEEKLY FACT
# ============================================================

weekly.to_parquet(
    WEEKLY_PARQUET,
    index=False
)

weekly.to_csv(
    WEEKLY_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 22. SUMMARY
# ============================================================

print("\n==========================================")
print("STEP 10 COMPLETED")
print("==========================================")

print("\nEMPLOYEE MASTER")
print("Unique employees:", len(employee_master))
print("Employee attributes:", len(employee_attribute_columns))

print("\nATTENDANCE REQUIRED")
print("Total employee-date rows:", len(attendance_required))
print(
    "Rows with attendance transaction:",
    int(attendance_required["attendance_record_count"].gt(0).sum())
)
print(
    "Rows without attendance transaction:",
    int(attendance_required["attendance_record_count"].eq(0).sum())
)

print("\nWEEKLY FACT")
print("Employee-week rows:", len(weekly))

print("\nCOST CENTER DISTRIBUTION")
print(
    weekly["cost_center_type"].value_counts(dropna=False)
)

print("\nWEEKLY REQUIRED DAYS DISTRIBUTION")
print(
    weekly["weekly_required_days"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nWEEKLY ATTENDANCE STATUS")
print(
    weekly["weekly_attendance_status"].value_counts(dropna=False)
)

print("\nOUTPUT FILES")
print("Employee master:", EMPLOYEE_PARQUET)
print("Attendance required:", REQUIRED_PARQUET)
print("Daily fact:", DAILY_PARQUET)
print("Weekly fact:", WEEKLY_PARQUET)

print("\n==========================================")
print("PROCESS 10 SUCCESSFULLY FINISHED")
print("==========================================")


Loading attendance and calendar data...
Attendance source rows: 27711
Calendar rows: 365

Employee master created.
Unique employees: 5729
Employee attributes retained: 57

                           column  employees_with_multiple_values
                            day_s                            5369
                            shift                             601
                 total_work_hours                            5111
                  attendance_type                              21
               enter_overtime_awh                             107
                exit_overtime_awh                             115
         total_overtime_hours_awf                             102
             total_overtime_hours                             102
                           spl_no                             111
                           status                            1925
                attendance_source                              79
             attendance_edit_date   

ValueError: has_attendance_activity is inconsistent with clock flags.